# SAS to Python: Finding Missing Rows

This notebook demonstrates how to debug missing rows after a data migration using set differences and `dataprof`.

In [1]:
# Run this cell first to install dependencies in Colab
!pip install pandas dataprof==0.7.0

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## Generate Dummy Data

Run the cell below to create `mother_db.csv` and `broken_subset_db.csv` for our exercise.

In [2]:
import pandas as pd
import numpy as np
import random

# Seed both RNGs so results are fully reproducible
np.random.seed(42)
random.seed(42)

# Create dummy 'mother_db'
num_rows = 1000
mother_data = {
    'contract_id': [f'C{i:05d}' for i in range(1, num_rows + 1)],
    'condition': np.random.choice([True, False], size=num_rows, p=[0.7, 0.3]),
    'value': np.random.normal(100, 15, size=num_rows),
    'YOUR_SAMPLE_COLUMN': [random.choice([np.nan, 'A', 'B', 'C']) for _ in range(num_rows)]
}
mother_db = pd.DataFrame(mother_data)
mother_db.to_csv("mother_db.csv", index=False)

# Create dummy 'broken_subset_db' (missing some rows, some NaNs)
subset_data = mother_db[mother_db['condition']].copy()
subset_data = subset_data.drop(subset_data.sample(40, random_state=42).index)
subset_data.loc[subset_data.sample(10, random_state=42).index, 'YOUR_SAMPLE_COLUMN'] = np.nan
subset_data.to_csv("broken_subset_db.csv", index=False)

print("Created dummy data: 'mother_db.csv' and 'broken_subset_db.csv'")
print(f"  mother_db:       {len(mother_db)} rows")
print(f"  broken_subset_db: {len(subset_data)} rows (condition=True minus 40 dropped)")

Created dummy data: 'mother_db.csv' and 'broken_subset_db.csv'
  mother_db:       1000 rows
  broken_subset_db: 672 rows (condition=True minus 40 dropped)


## Debugging the Missing Rows

Find which rows disappeared using set differences, then profile both datasets with `dataprof` to understand whether the missing rows introduce any statistical bias.

In [3]:
import pandas as pd
import dataprof as dp

# Load your dataframes
mother_db = pd.read_csv("mother_db.csv")
broken_subset_db = pd.read_csv("broken_subset_db.csv")

# 1. Get the IDs that SHOULD be in the perimeter (from the mother DB)
expected_ids = set(mother_db[mother_db['condition']]['contract_id'])

# 2. Get the IDs that actually MADE IT into the new sub-DB
actual_ids = set(broken_subset_db['contract_id'])

# 3. Find the missing ones
missing_ids = expected_ids - actual_ids
print(f"Missing {len(missing_ids)} IDs out of {len(expected_ids)} expected\n")

# 4. Print the exact rows that vanished — look for patterns
missing_rows = mother_db[mother_db['contract_id'].isin(missing_ids)]
print("Vanished rows sample (first 10):")
display(missing_rows.head(10))

# Quick sanity check: are missing rows biased by value?
kept_rows = mother_db[mother_db['contract_id'].isin(actual_ids)]
print(f"\nValue stats — missing rows : mean={missing_rows['value'].mean():.2f}, std={missing_rows['value'].std():.2f}")
print(f"Value stats — kept rows   : mean={kept_rows['value'].mean():.2f},   std={kept_rows['value'].std():.2f}")
print("(Random drop → no systematic bias expected. Profiling will confirm.)\n")

print("=" * 50)

# 5. Profile both datasets — name= makes quality_summary() readable
report_mother = dp.profile(mother_db, name="mother_db")
report_subset = dp.profile(broken_subset_db, name="broken_subset_db")

print("--- MOTHER DB PROFILE ---")
print(f"Total rows: {report_mother.rows}")

# Dict-like column access
sample_col = report_mother["YOUR_SAMPLE_COLUMN"]
print(f"Nulls in 'YOUR_SAMPLE_COLUMN': {sample_col.null_count} / {report_mother.rows} ({sample_col.null_percentage:.1f}%)")

print("\n--- SUBSET DB PROFILE ---")
print(f"Total rows: {report_subset.rows}")

sample_col_sub = report_subset["YOUR_SAMPLE_COLUMN"]
print(f"Nulls in 'YOUR_SAMPLE_COLUMN': {sample_col_sub.null_count} / {report_subset.rows} ({sample_col_sub.null_percentage:.1f}%)")

# 6. Side-by-side statistical summary
print("\n--- STATISTICAL SUMMARY: mother_db ---")
print(report_mother.describe())

print("\n--- STATISTICAL SUMMARY: broken_subset_db ---")
print(report_subset.describe())

# 7. Quality comparison across both datasets
print("\n--- QUALITY COMPARISON ---")
comparison = pd.DataFrame([report_mother.quality_summary(), report_subset.quality_summary()])
print(comparison[["source", "rows", "quality_score", "completeness", "uniqueness"]])

# 8. Export profiles — JSON for diffing, CSV for quick spreadsheet review
report_mother.save("profile_mother.json").save("profile_mother.csv")
report_subset.save("profile_subset.json").save("profile_subset.csv")
print("\nSaved profiles to JSON and CSV.")

Missing 40 IDs out of 712 expected

Vanished rows sample (first 10):


,contract_id,condition,value,YOUR_SAMPLE_COLUMN
54,C00055,True,99.479727,A
66,C00067,True,104.337530,A
90,C00091,True,97.345792,A
95,C00096,True,126.918368,A
117,C00118,True,102.200705,A
123,C00124,True,119.176778,C
125,C00126,True,100.696548,NaN
168,C00169,True,99.316210,A
170,C00171,True,96.381459,A
184,C00185,True,98.653965,C



Value stats — missing rows : mean=102.31, std=13.57
Value stats — kept rows   : mean=102.27,   std=14.76
(Random drop → no systematic bias expected. Profiling will confirm.)

--- MOTHER DB PROFILE ---
Total rows: 1000
Nulls in 'YOUR_SAMPLE_COLUMN': 243 / 1000 (24.3%)

--- SUBSET DB PROFILE ---
Total rows: 672
Nulls in 'YOUR_SAMPLE_COLUMN': 168 / 672 (25.0%)

--- STATISTICAL SUMMARY: mother_db ---
             condition  contract_id  YOUR_SAMPLE_COLUMN      value
count         1000.000       1000.0              1000.0  1000.0000
null%            0.000          0.0                24.3     0.0000
unique           2.000       1000.0                 3.0  1000.0000
mean               NaN          NaN                 NaN   101.4834
std                NaN          NaN                 NaN    14.8340
min                NaN          NaN                 NaN    56.1797
25%                NaN          NaN                 NaN    91.5400
50%                NaN          NaN                 NaN   101.2